# Shipping Voyage Monte Carlo — Asia → Northern Europe

**Question:** What is the probability that a westbound Asia–Northern Europe container voyage leg (reference trade: Shanghai → Rotterdam) produces a negative voyage result?

**Scope (V1):** Westbound leg Shanghai → Yantian → Tanjung Pelepas → Rotterdam, time-charter perspective.
The output is a **voyage result** (revenue minus voyage-related costs), not company profit. Overhead/SG&A is excluded.

**How this notebook is organized**

| # | Section | What happens |
|---|---|---|
| 0 | Setup | Imports, settings, data URLs |
| 1 | Assumptions | All hardcoded numbers come from `assumptions.csv` |
| 2 | Processed data | Brent returns (later: FBX11, MRV) |
| 3 | Model | One function: `voyage_result()` |
| 4 | Deterministic baseline | Run the model once and sanity-check it |
| 5 | Uncertainty inputs | One sampling function per stochastic variable |
| 6 | Simulation | 100,000 scenarios |
| 7 | Results | P(loss), percentiles, break-even |
| 8 | Sensitivity | Which inputs drive the result |
| 9 | Route scenarios | Suez vs. Cape |

Data analysis lives in separate notebooks (`01_fuel_price`, `02_fuel_consumption`, `03_freight_rate`). This notebook only **reads processed data**.

## 0. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Simulation settings ----
SEED = 42
N_SIMULATIONS = 100_000
ROUTE = "suez"            # "suez" or "cape"

# Turn uncertainty on one variable at a time.
# False = the variable is held at its baseline value.
STOCHASTIC = {
    "freight_rate": True,
    "fuel_price":   True,
    "utilization":  True,
    "delay_days":   True,
}

# ---- Data locations (raw GitHub URLs) ----
# Replace <user>/<repo> with your repository.
BASE_URL = "https://raw.githubusercontent.com/<user>/<repo>/main"
ASSUMPTIONS_URL = f"{BASE_URL}/assumptions.csv"
BRENT_URL       = f"{BASE_URL}/data/processed/brent_monthly_processed.csv"

rng = np.random.default_rng(SEED)

In [ ]:
def load_csv(url, local_name):
    """Use a local copy if it exists (handy while testing), otherwise read from GitHub."""
    if os.path.exists(local_name):
        return pd.read_csv(local_name)
    if "<user>" in url:
        raise ValueError(f"Set BASE_URL in the setup cell before loading {local_name}.")
    return pd.read_csv(url)

## 1. Assumptions

Every fixed number in the model lives in `assumptions.csv` with its unit, type, status, source and date.
**Type** is one of `data`, `proxy`, `synthetic`, `hardcoded`, `simplification`.
**Status** is `final` or `placeholder`. The goal is to turn every placeholder into a documented value.

In [ ]:
assumptions_df = load_csv(ASSUMPTIONS_URL, "assumptions.csv")
A = dict(zip(assumptions_df["variable"], assumptions_df["value"].astype(float)))

n_placeholders = (assumptions_df["status"] == "placeholder").sum()
print(f"{len(A)} assumptions loaded — {n_placeholders} still placeholders")
assumptions_df[["variable", "value", "unit", "type", "status"]]

## 2. Processed data

**Brent monthly returns** — historical proxy for movements in VLSFO prices (see `01_fuel_price.ipynb`).
Assumption: % change in VLSFO = % change in Brent (1:1 pass-through, synthetic assumption).

In [ ]:
brent_df = load_csv(BRENT_URL, "brent_monthly_processed.csv")
brent_returns = brent_df["monthly_return"].dropna().to_numpy()

print(f"Brent returns: {len(brent_returns)} months | "
      f"mean {brent_returns.mean():+.2%} | std {brent_returns.std(ddof=1):.2%} | "
      f"min {brent_returns.min():+.2%} | max {brent_returns.max():+.2%}")

## 3. Model

One function. It works for single numbers (baseline) **and** for arrays of 100,000 values (simulation), because NumPy handles both the same way.

**Revenue** = freight rate × loaded FEU  
**Costs** = fuel + charter + port dues + canal + cargo handling + equipment + EU ETS

Days are split into **sea days** (main-engine fuel) and **port/delay days** (auxiliary fuel only). Charter is paid for all days.

In [ ]:
def route_params(A, route):
    """Baseline inputs for a given routing."""
    if route not in ("suez", "cape"):
        raise ValueError("route must be 'suez' or 'cape'")
    return {
        "freight_rate": A["freight_rate_baseline"],
        "fuel_price":   A["vlsfo_baseline"],
        "utilization":  A["utilization_mode"],
        "delay_days":   A["delay_mode"],
        "sea_days":     A["sea_days_suez"] if route == "suez" else A["sea_days_cape"],
        "canal_cost":   A["suez_toll"] if route == "suez" else 0.0,
    }


def voyage_result(inputs, A):
    """Voyage result for one or many scenarios. Returns a dict of components."""
    freight_rate = np.asarray(inputs["freight_rate"], dtype=float)
    fuel_price   = np.asarray(inputs["fuel_price"],   dtype=float)
    utilization  = np.asarray(inputs["utilization"],  dtype=float)
    delay_days   = np.asarray(inputs["delay_days"],   dtype=float)
    sea_days     = inputs["sea_days"]
    canal_cost   = inputs["canal_cost"]

    # Cargo and revenue
    loaded_feu = A["capacity_teu"] * utilization / A["teu_per_feu"]
    revenue = freight_rate * loaded_feu

    # Time
    port_days  = A["port_days"] + delay_days
    total_days = sea_days + port_days

    # Fuel
    fuel_sea_mt  = sea_days * A["fuel_cons_sea"]
    fuel_port_mt = port_days * A["fuel_cons_port"]
    fuel_cost = (fuel_sea_mt + fuel_port_mt) * fuel_price

    # Other voyage costs
    charter_cost  = A["charter_rate"] * total_days
    port_dues     = (A["port_cost_shanghai"] + A["port_cost_yantian"]
                     + A["port_cost_tanjungpelepas"] + A["port_cost_rotterdam"])
    handling_cost  = loaded_feu * A["moves_per_feu"] * A["handling_cost_per_move"]
    equipment_cost = loaded_feu * A["equipment_cost_per_feu"]

    # EU ETS: only the sea fuel on the leg into the EU, 50% coverage
    ets_co2  = fuel_sea_mt * A["ets_leg_share"] * A["co2_factor"]
    ets_cost = ets_co2 * A["ets_coverage"] * A["ets_phase_in"] * A["eua_price_usd"]

    total_cost = (fuel_cost + charter_cost + port_dues + canal_cost
                  + handling_cost + equipment_cost + ets_cost)

    return {
        "loaded_feu": loaded_feu,
        "revenue": revenue,
        "fuel_cost": fuel_cost,
        "charter_cost": charter_cost,
        "port_dues": port_dues,
        "canal_cost": canal_cost,
        "handling_cost": handling_cost,
        "equipment_cost": equipment_cost,
        "ets_cost": ets_cost,
        "total_cost": total_cost,
        "result": revenue - total_cost,
        "total_days": total_days,
    }

## 4. Deterministic baseline

Run the model once with baseline values. **Sanity check:** a voyage margin far above what liner companies report (typically single digits to low tens of percent outside boom years) means costs are missing or inputs are off.

In [ ]:
base_inputs = route_params(A, ROUTE)
base = voyage_result(base_inputs, A)

cost_items = ["fuel_cost", "charter_cost", "port_dues", "canal_cost",
              "handling_cost", "equipment_cost", "ets_cost"]

baseline_table = pd.DataFrame({
    "USD": [base["revenue"]] + [base[c] for c in cost_items] + [base["total_cost"], base["result"]],
}, index=["revenue"] + cost_items + ["total_cost", "voyage_result"])
baseline_table["% of revenue"] = baseline_table["USD"] / base["revenue"] * 100

margin = base["result"] / base["revenue"]
breakeven_freight = base["total_cost"] / base["loaded_feu"]

print(f"Route: {ROUTE} | Voyage days: {float(base['total_days']):.0f} | Loaded FEU: {float(base['loaded_feu']):,.0f}")
print(f"Margin: {margin:.1%} | Break-even freight rate: ${float(breakeven_freight):,.0f}/FEU")
if margin > 0.30:
    print("⚠ Margin looks high. Check for missing costs or optimistic inputs.")

baseline_table.style.format({"USD": "{:,.0f}", "% of revenue": "{:.1f}"})

## 5. Uncertainty inputs

One function per stochastic variable. Each returns an array of length `n`.

| Variable | Method | Data type |
|---|---|---|
| Freight rate | **TEMPORARY** lognormal around baseline → replace with FBX11 analysis | synthetic (placeholder) |
| Fuel price | VLSFO baseline × (1 + bootstrapped Brent monthly return) | historical proxy |
| Utilization | Triangular (min, mode, max) | synthetic |
| Delay days | Triangular (min, mode, max) | synthetic |

Known limitation (V1): variables are sampled independently. Freight rate and fuel price are correlated in reality (e.g. bunker surcharges). This is addressed once FBX11 data is available.

In [ ]:
def sample_freight_rate(n, rng, A):
    # PLACEHOLDER: median = baseline, log-sd from assumptions
    return A["freight_rate_baseline"] * np.exp(rng.normal(0.0, A["freight_rate_sigma"], n))

def sample_fuel_price(n, rng, A, brent_returns):
    sampled_returns = rng.choice(brent_returns, size=n, replace=True)
    return A["vlsfo_baseline"] * (1 + sampled_returns)

def sample_utilization(n, rng, A):
    return rng.triangular(A["utilization_min"], A["utilization_mode"], A["utilization_max"], n)

def sample_delay_days(n, rng, A):
    return rng.triangular(A["delay_min"], A["delay_mode"], A["delay_max"], n)


def build_inputs(n, rng, A, route, stochastic):
    """Baseline values for everything, replaced by samples where stochastic[var] is True."""
    base = route_params(A, route)
    samplers = {
        "freight_rate": lambda: sample_freight_rate(n, rng, A),
        "fuel_price":   lambda: sample_fuel_price(n, rng, A, brent_returns),
        "utilization":  lambda: sample_utilization(n, rng, A),
        "delay_days":   lambda: sample_delay_days(n, rng, A),
    }
    inputs = dict(base)
    for var, sampler in samplers.items():
        inputs[var] = sampler() if stochastic[var] else np.full(n, base[var])
    return inputs

## 6. Simulation

In [ ]:
sim_inputs = build_inputs(N_SIMULATIONS, rng, A, ROUTE, STOCHASTIC)
sim = voyage_result(sim_inputs, A)

results_df = pd.DataFrame({
    "freight_rate": sim_inputs["freight_rate"],
    "fuel_price":   sim_inputs["fuel_price"],
    "utilization":  sim_inputs["utilization"],
    "delay_days":   sim_inputs["delay_days"],
    "revenue":      sim["revenue"],
    "total_cost":   sim["total_cost"],
    "result":       sim["result"],
})

stochastic_on = [k for k, v in STOCHASTIC.items() if v]
print(f"{N_SIMULATIONS:,} scenarios | route: {ROUTE} | stochastic: {', '.join(stochastic_on) or 'none'}")
results_df.describe().T.style.format("{:,.2f}")

## 7. Results

In [ ]:
r = results_df["result"]

summary = pd.Series({
    "Mean result (USD)":       r.mean(),
    "Median result (USD)":     r.median(),
    "Std. deviation (USD)":    r.std(),
    "5th percentile (USD)":    r.quantile(0.05),
    "95th percentile (USD)":   r.quantile(0.95),
    "P(result < 0)":           (r < 0).mean(),
})
print(summary.to_string(float_format=lambda x: f"{x:,.3f}" if abs(x) < 1 else f"{x:,.0f}"))

# Break-even values (with all other inputs at baseline)
be_freight = base["total_cost"] / base["loaded_feu"]
cost_per_feu_variable = A["moves_per_feu"] * A["handling_cost_per_move"] + A["equipment_cost_per_feu"]
fixed_cost = base["total_cost"] - base["loaded_feu"] * cost_per_feu_variable
be_utilization = fixed_cost / (A["capacity_teu"] / A["teu_per_feu"]
                               * (A["freight_rate_baseline"] - cost_per_feu_variable))
print(f"\nBreak-even freight rate:  ${float(be_freight):,.0f}/FEU")
print(f"Break-even utilization:   {float(be_utilization):.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(r / 1e6, bins=150, color="steelblue", alpha=0.85)
ax.axvline(0, color="crimson", lw=1.5, label=f"Break-even (P(loss) = {(r < 0).mean():.1%})")
ax.axvline(r.median() / 1e6, color="black", ls="--", lw=1, label="Median")
ax.set_xlabel("Voyage result (USD million)")
ax.set_ylabel("Number of scenarios")
ax.set_title(f"Distribution of voyage result — {ROUTE.title()} routing, {N_SIMULATIONS:,} scenarios")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Sensitivity

Spearman rank correlation between each stochastic input and the voyage result. A larger absolute value means the input drives more of the variation.

In [ ]:
inputs_on = [k for k, v in STOCHASTIC.items() if v]
sensitivity = (results_df[inputs_on + ["result"]]
               .corr(method="spearman")["result"]
               .drop("result")
               .sort_values(key=abs, ascending=True))

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(sensitivity.index, sensitivity.values,
        color=["seagreen" if v > 0 else "indianred" for v in sensitivity.values])
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Spearman correlation with voyage result")
ax.set_title("What drives the voyage result?")
plt.tight_layout()
plt.show()
sensitivity.sort_values(key=abs, ascending=False).round(3)

## 9. Route scenarios — Suez vs. Cape

Same model and same uncertainty, run once per routing. Uses a fresh RNG with the same seed so the comparison is fair.

In [ ]:
scenario_rows = []
for route in ["suez", "cape"]:
    rng_s = np.random.default_rng(SEED)
    res = voyage_result(build_inputs(N_SIMULATIONS, rng_s, A, route, STOCHASTIC), A)["result"]
    scenario_rows.append({
        "route": route,
        "median_result": np.median(res),
        "p5": np.percentile(res, 5),
        "p95": np.percentile(res, 95),
        "P(loss)": (res < 0).mean(),
    })

pd.DataFrame(scenario_rows).set_index("route").style.format({
    "median_result": "{:,.0f}", "p5": "{:,.0f}", "p95": "{:,.0f}", "P(loss)": "{:.1%}"})

---
**Status:** All results above use placeholder inputs where `status = placeholder` in `assumptions.csv`. They are for checking the model mechanics and prioritizing data work, **not** for drawing conclusions.